# 27.2 Neuromorphic computing

Neuromorphic learning treats time and sparse spikes as computation, not implementation detail. Recurrent voltage carries evidence, thresholds emit events, and sparse activity can skip dense work. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(272)


def accuracy(y_true, y_pred):
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def active_fraction(spikes):
    return float(np.mean(np.asarray(spikes) > 0))

## The concept, built once (D1)

A leaky integrate-and-fire neuron uses $$\tilde v_t=\lambda v_{t-1}+x_t,\quad s_t=\mathbf{1}[\tilde v_t\ge\theta],\quad v_t=(1-s_t)\tilde v_t.$$ With $x_1=1.2$, $\lambda=0.8$, and $\theta=1$, the lesson spike train is $[1,0,0,0,0]$.

In [ ]:
def lif_run(inputs, leak, threshold, reset=True):
    voltage = 0.0
    voltages = []
    spikes = []
    pre_reset = []
    for x in inputs:
        candidate = leak * voltage + float(x)
        spike = int(candidate >= threshold)
        if reset and spike:
            voltage = 0.0
        else:
            voltage = candidate
        pre_reset.append(candidate)
        voltages.append(voltage)
        spikes.append(spike)
    return np.array(voltages), np.array(spikes), np.array(pre_reset)

strong_inputs = np.array([1.2, 0.0, 0.0, 0.0, 0.0])
strong_v, strong_s, strong_pre = lif_run(strong_inputs, leak=0.8, threshold=1.0)

assert strong_s.tolist() == [1, 0, 0, 0, 0]
assert round(float(strong_pre[0]), 3) == 1.200

print("strong spike train", strong_s.tolist())

Weak evidence accumulates through the leak term. Five inputs of $0.3$ with $\lambda=0.8$ reach $v_5=1.008$ before reset and spike once.

In [ ]:
weak_inputs = np.repeat(0.3, 5)
weak_v, weak_s, weak_pre = lif_run(weak_inputs, leak=0.8, threshold=1.0)
slow_v, slow_s, slow_pre = lif_run(np.repeat(0.3, 4), leak=0.2, threshold=10.0, reset=False)
fast_v, fast_s, fast_pre = lif_run(np.repeat(0.3, 4), leak=0.8, threshold=10.0, reset=False)
weighted_input = float(np.dot(np.array([1, 0, 1]), np.array([0.6, 0.2, 0.5])))
active = 12 / 1000
skipped = 1 - active

assert weak_s.tolist() == [0, 0, 0, 0, 1]
assert round(float(weak_pre[-1]), 3) == 1.008
assert round(float(slow_pre[-1]), 3) == 0.374
assert round(float(fast_pre[-1]), 3) == 0.886
assert round(weighted_input, 3) == 1.100
assert round(skipped, 3) == 0.988

print("weak pre-reset voltage", np.round(weak_pre, 3))
print("weighted input", round(weighted_input, 3))
print("skipped fraction", round(skipped, 3))

## The dataset ladder

The ladder rises from a single LIF neuron to sparse bursts, noisy weak evidence, a small event-camera grid, and a dense badly scaled stream where low thresholds destroy sparsity.

In [ ]:
def make_neuro_ladder(seed=272):
    local_rng = np.random.default_rng(seed)
    ladder = []

    ladder.append({"name": "D1 single neuron", "streams": np.array([[1.2, 0, 0, 0, 0], [0.3, 0.3, 0.3, 0.3, 0.3]]), "labels": np.array([1, 1]), "leak": 0.8, "threshold": 1.0})

    clean = np.zeros((12, 20))
    labels = np.zeros(12, dtype=int)
    for i in range(12):
        labels[i] = i % 2
        start = 3 if labels[i] == 0 else 12
        clean[i, start:start + 3] = 0.55
    ladder.append({"name": "D2 sparse bursts", "streams": clean, "labels": labels, "leak": 0.75, "threshold": 1.0})

    noisy = local_rng.normal(0.16, 0.06, size=(20, 30))
    labels = np.zeros(20, dtype=int)
    for i in range(20):
        labels[i] = i % 2
        window = slice(8, 17) if labels[i] == 1 else slice(0, 5)
        noisy[i, window] += 0.16
    noisy = np.clip(noisy, 0.0, None)
    ladder.append({"name": "D3 weak noisy events", "streams": noisy, "labels": labels, "leak": 0.85, "threshold": 1.0})

    grid = local_rng.binomial(1, 0.025, size=(18, 8, 8, 16)).astype(float)
    labels = np.zeros(18, dtype=int)
    for i in range(18):
        labels[i] = i % 2
        if labels[i] == 1:
            grid[i, 3:5, 3:5, 5:10] += 1.0
        else:
            grid[i, 1:3, 1:3, 1:5] += 1.0
    streams = grid.reshape(18, 64, 16).sum(axis=1) / 6.0
    ladder.append({"name": "D4 event grid", "streams": streams, "labels": labels, "leak": 0.82, "threshold": 1.0})

    dense = local_rng.normal(0.38, 0.12, size=(24, 50))
    labels = np.zeros(24, dtype=int)
    for i in range(24):
        labels[i] = i % 2
        if labels[i] == 1:
            dense[i, 20:35] += 0.20
    dense = np.clip(dense, 0.0, None)
    ladder.append({"name": "D5 dense bad scaling", "streams": dense, "labels": labels, "leak": 0.9, "threshold": 0.45})

    return ladder

neuro_ladder = make_neuro_ladder()

for rung in neuro_ladder:
    print(rung["name"], "shape", rung["streams"].shape, "labels", np.bincount(rung["labels"], minlength=2).tolist(), "sample", np.round(rung["streams"][0, :6], 3))

In [ ]:
def run_neuro_rung(rung):
    spike_rows = []
    voltage_rows = []
    scores = []
    for stream in rung["streams"]:
        voltages, spikes, pre = lif_run(stream, rung["leak"], rung["threshold"])
        spike_rows.append(spikes)
        voltage_rows.append(pre)
        midpoint = len(spikes) // 2
        score = spikes[midpoint:].sum() - spikes[:midpoint].sum()
        scores.append(score)
    scores = np.asarray(scores)
    pred = (scores > 0).astype(int)
    spikes = np.asarray(spike_rows)
    voltages = np.asarray(voltage_rows)
    acc = accuracy(rung["labels"], pred)
    frac = active_fraction(spikes)
    utility = acc / max(frac, 0.01)
    return {"accuracy": acc, "active_fraction": frac, "utility": utility, "spikes": spikes, "voltages": voltages, "pred": pred}

neuro_results = []

for rung in neuro_ladder:
    result = run_neuro_rung(rung)
    neuro_results.append(result)
    print(f"{rung['name']}: accuracy={result['accuracy']:.3f} active_fraction={result['active_fraction']:.3f} utility={result['utility']:.3f}")

print("no-skill D5 majority baseline", round(float(max(np.mean(neuro_ladder[-1]["labels"] == 0), np.mean(neuro_ladder[-1]["labels"] == 1))), 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))

for i, rung in enumerate(neuro_ladder):
    result = neuro_results[i]
    ax = axes[0, i]
    if result["spikes"].ndim == 2:
        ax.imshow(result["spikes"][:8], aspect="auto", cmap="Greys")
    ax.set_title(rung["name"])
    ax.set_xlabel("time")
    ax.set_ylabel("sample")

x = np.arange(1, 6)
utilities = [r["utility"] for r in neuro_results]
fractions = [r["active_fraction"] for r in neuro_results]
axes[1, 0].plot(x, utilities, marker="o", label="accuracy / active fraction")
axes[1, 0].set_title("Utility vs complexity")
axes[1, 0].set_xlabel("D rung")
axes[1, 0].legend()

for j in range(1, 5):
    axes[1, j].bar(["active"], [fractions[j]])
    axes[1, j].set_ylim(0, 1)
    axes[1, j].set_title(f"D{j + 1} active fraction")

plt.tight_layout()

## Pitfall on the hardest rung: assuming sparsity automatically appears

D5 uses a dense stream and too-low threshold, so the active fraction explodes and the energy advantage disappears. Normalize inputs and raise the threshold to recover sparse useful spikes.

In [ ]:
d5 = neuro_ladder[-1]
wrong = run_neuro_rung(d5)
fixed = dict(d5)
fixed["streams"] = d5["streams"] / np.percentile(d5["streams"], 95)
fixed["threshold"] = 1.0
fixed["leak"] = 0.85
right = run_neuro_rung(fixed)

print("bad active fraction", round(wrong["active_fraction"], 3), "accuracy", round(wrong["accuracy"], 3))
print("fixed active fraction", round(right["active_fraction"], 3), "accuracy", round(right["accuracy"], 3))

## Evaluate it + Practice
- Compare the rung metric against the no-skill baseline printed above.
- Run one cheap sanity check: change one label, threshold, route, or floor and confirm the metric moves in the expected direction.
- Ablate the key idea by turning off the kernel, leak, tool verification, feedback, or safety gate.
- Watch failure signals: flat metric curves, hidden weak coordinates, high cost, or a D5 pitfall that does not improve after the fix.

Practice prompts:
1. Change the D3 noise or shift level and predict which rung changes most.

2. Replace the baseline with a stronger classical or rule-based check and compare the metric.

3. Add one new D5 stress case and explain whether the pitfall fix still works.